# CEG-WM Content V8 — unified Drive-first formal handoff

This operational Notebook mounts Google Drive first, clones Content-V8, records the actual branch/commit/clean state, installs only the checked-out project, reads CEG_WM_ROOT_KEY and HF_TOKEN from Colab Secrets, and invokes experiments.run_content_v8_formal_initial exactly once.

Each fresh runtime writes beneath /content/drive/MyDrive/CEG-WM/Content/Content-V8-short-commit-UTC/. A manual rerun must start from a fresh Colab runtime and therefore receives a new UTC directory. There is no automatic retry, resume, or fallback. This Notebook does not interpret scientific outcomes.


## 1. Mount Google Drive


In [ ]:
import json
from google.colab import drive

HANDOFF_FAILED = False
RUNNER_ATTEMPTED = False
FAILURE_PREFIX = "CEGWM_CONTENT_V8_FORMAL_HANDOFF_FAILURE"
_ALLOWED_ERRORS = {
    "CalledProcessError", "FileExistsError", "ImportError", "MemoryError",
    "ModuleNotFoundError", "OSError", "OutOfMemoryError", "RuntimeError",
    "TimeoutError", "TypeError", "UnicodeDecodeError", "ValueError",
}

def fail(stage, error_class="RuntimeError"):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    HANDOFF_FAILED = True
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    payload = {"status": "operational_failure", "stage": stage, "error_class": error_class}
    line = FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":"))
    if len(line.encode("utf-8")) <= 4096:
        print(line, flush=True)

try:
    drive.mount("/content/drive")
except BaseException as error:
    fail("drive_mount", type(error).__name__)


## 2. Fresh clone and record checkout identity


In [ ]:
import pathlib
import subprocess
import sys
from datetime import datetime, timezone

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
VERSION_ID = "Content-V8"
BRANCH = "Content-V8"
RUNNER_MODULE = "experiments.run_content_v8_formal_initial"
RUN_ID = "content-v8-bd6269861412-0ba01f405106"
repo = pathlib.Path("/content/cegwm-content-v8-source")
content_root = pathlib.Path("/content/drive/MyDrive/CEG-WM/Content")

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True,
        capture_output=True, text=True,
    ).stdout.strip()

if not HANDOFF_FAILED:
    try:
        if repo.exists():
            raise FileExistsError
        subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        execution_branch = git("branch", "--show-current")
        execution_exact = git("rev-parse", "HEAD")
        execution_clean = git("status", "--porcelain") == ""
        execution_short = execution_exact[:12]
        run_utc = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        run_root = content_root / f"{VERSION_ID}-{execution_short}-{run_utc}"
        content_root.mkdir(parents=True, exist_ok=True)
        if run_root.exists():
            raise FileExistsError
        identity = {
            "status": "checkout_recorded",
            "version": VERSION_ID,
            "branch": execution_branch,
            "commit": execution_exact,
            "short_commit": execution_short,
            "clean": execution_clean,
            "utc": run_utc,
            "drive_run_root": str(run_root),
        }
        line = "CEGWM_CONTENT_V8_FORMAL_IDENTITY" + " " + json.dumps(identity, sort_keys=True, separators=(",", ":"))
        if len(line.encode("utf-8")) > 4096:
            raise RuntimeError
        print(line, flush=True)
    except BaseException as error:
        fail("source_checkout_and_record", type(error).__name__)


## 3. Install checkout, read Secrets, and invoke the runner once


In [ ]:
import os
from google.colab import userdata

if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    root_key = ""
    hf_token = ""
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_error = None
    CAPTURE_LIMIT = 8192
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        root_key = userdata.get("CEG_WM_ROOT_KEY")
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(root_key, str) or not root_key.strip():
            raise RuntimeError
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["CEG_WM_ROOT_KEY"] = root_key
        runner_env["HF_TOKEN"] = hf_token
        root_key = ""
        hf_token = ""
        process = subprocess.Popen(
            [
                sys.executable, "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", execution_exact,
                "--artifact-sink", str(run_root),
            ],
            cwd=repo, env=runner_env,
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        )
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, CAPTURE_LIMIT - len(captured))
            captured.extend(chunk[:remaining])
            if len(chunk) > remaining:
                capture_overflow = True
        runner_rc = process.wait()
    except BaseException as error:
        launch_error = type(error).__name__
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        root_key = ""
        hf_token = ""
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
        runner_env = None
        captured.clear()

    if launch_error is not None:
        fail("formal_runner_launch", launch_error)
    elif runner_rc != 0:
        fail("formal_runner_nonzero", "CalledProcessError")
    elif capture_overflow:
        fail("formal_runner_stdout_overflow", "RuntimeError")


## 4. Check existing Drive artifacts only


In [ ]:
import re


run_dir = run_root / RUN_ID / "terminal"
terminal_archive = run_dir / (RUN_ID + ".zip")
terminal_sidecar = run_dir / (RUN_ID + ".zip.sha256")

def validate_pair(archive_path, sidecar_path):
    if not archive_path.is_file() or not sidecar_path.is_file():
        raise RuntimeError
    if sidecar_path.stat().st_size > 1024:
        raise RuntimeError
    sidecar_text = sidecar_path.read_text(encoding="ascii")
    match = re.fullmatch(r"([0-9a-f]{64})  ([^\s]+)\n", sidecar_text)
    if match is None or match.group(2) != archive_path.name:
        raise RuntimeError
    return {
        "archive_path": str(archive_path),
        "sidecar_path": str(sidecar_path),
        "sha256": match.group(1),
    }

if not HANDOFF_FAILED:
    try:

        if not terminal_archive.is_file() or not terminal_sidecar.is_file():
            raise RuntimeError
        artifact_kind = "terminal"
        pairs = [validate_pair(terminal_archive, terminal_sidecar)]
        receipt = {
            "status": "drive_artifacts_ready",
            "version": VERSION_ID,
            "branch": execution_branch,
            "commit": execution_exact,
            "utc": run_utc,
            "drive_run_root": str(run_root),
            "artifact_kind": artifact_kind,
            "pairs": pairs,
        }
        line = "CEGWM_CONTENT_V8_FORMAL_ARTIFACT" + " " + json.dumps(receipt, sort_keys=True, separators=(",", ":"))
        if len(line.encode("utf-8")) > 4096:
            raise RuntimeError
        print(line, flush=True)
    except BaseException as error:
        fail("artifact_pair_validation", type(error).__name__)


## Stop boundary

Return the bounded identity record and Drive artifact receipt, or the single sanitized failure line. The final cell never invokes the runner. Do not expose child stdout/stderr, Secrets, prompts, keys, tokens, or private runtime state. Do not interpret the resulting evidence package as a scientific decision.
